# basic 

In [1]:
%load_ext autoreload
%autoreload all

In [2]:
import polars as pl
import pickle


import src.configs_new_mapped as configs_mapped
import src.graph_fct as graph_fct
import src.new_tokenizer as new_tokenizer

# 1. prepare mapped concepts and subgraph

In [10]:
df_relations = pl.read_parquet(f"{configs_mapped.GraphConfig().relation_path}")
df_mapped = pl.read_parquet(f"{configs_mapped.GraphConfig().concept_path}").filter(pl.col("is_mapped")).unique()

In [11]:
mapped_ids = df_mapped["id"].to_list()
whole_graph = graph_fct.build_relations_graph(df_relations, col_src="src.id", col_dst="dst.id", col_relation="relation")
combined_subgraphs = graph_fct.get_combined_subgraphs_from_nodes(whole_graph, mapped_ids, max_distance=configs_mapped.TokenizerParam().max_dist_candidate)

In [12]:
len(combined_subgraphs.nodes()), len(combined_subgraphs.edges()), list(combined_subgraphs.edges(data=True))[:5]



(75635,
 290762,
 [('56923001', '36504009', {'relation': 'ASSOCIATED_MORPHOLOGY'}),
  ('56923001', '896913006', {'relation': 'IS_A'}),
  ('56923001', '40238009', {'relation': 'FINDING_SITE'}),
  ('56923001', '302154007', {'relation': 'IS_A'}),
  ('56923001', '428360003', {'relation': 'IS_A'})])

# 2. tokenizer apply test

In [13]:
# save
with open(configs_mapped.ProcessedGraph().combined_subgraphs, "wb") as f:
    pickle.dump(combined_subgraphs, f)

df_mapped.write_parquet(configs_mapped.ProcessedGraph().mapped_cpts)

In [17]:
with open(configs_mapped.ProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)
df = pl.read_parquet(configs_mapped.GraphConfig().concept_path).select("id", "label")
id_to_label = dict(zip(df["id"], df["label"]))

In [16]:
new_tokenizer.tokenize_all_rel("392091000", combined_subgraphs, ["225365006", "128927009", "129265001"], 3, id_to_label)
# "386053000"

(128927009, HAS_FOCUS(225365006), METHOD(129265001))

# semantic coverage matrix compuation

In [7]:
with open(configs_mapped.ProcessedGraph().combined_subgraphs, "rb") as f:
    combined_subgraphs = pickle.load(f)


df_mapped = pl.read_parquet(configs_mapped.ProcessedGraph().mapped_cpts)
mapped_ids = df_mapped["id"].to_list()



In [22]:
combined_subgraphs.out_edges("128927009")

OutMultiEdgeDataView([('128927009', '129264002'), ('128927009', '71388002')])

In [24]:
T = ["138875005", "404684003"]
S, node_to_idx = new_tokenizer.compute_semantic_coverage(combined_subgraphs, T, D=3)
score = new_tokenizer.semantic_coverage_score(S, node_to_idx, mapped_ids)
score

0.08080104027313023

# GREEDY SELECTION

In [ ]:
selector = new_tokenizer.LazyGreedyTokenSelector(combined_subgraphs, mapped_ids, D=3)
history = selector.select(k=len(mapped_ids))   # [(token, marginal_gain, cumulative_score), ...]
T = selector.T